# Simple Training & Generation Test of our Mamba LLM

### Setup
As per https://github.com/state-spaces/mamba/tree/main?tab=readme-ov-file

In [ ]:
%pip install dotenv datasets transformers

In [ ]:
## ONLY RUN IF ON COLAB

# import os, sys

# if not os.path.exists("csci739-project-mamba"):
#     os.system("git clone https://github.com/JackMclaughlin424/csci739-project-mamba.git")
# else:
#     os.system("cd csci739-project-mamba && git pull")

# sys.path.insert(0, "csci739-project-mamba")



In [5]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer
from datasets import load_dataset


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

Building model

In [6]:
from mamba.mamba_llm import MambaLMConfig, MambaLMHeadModel
import math

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

config = MambaLMConfig(
    d_input=256,
    d_model=512,          # 2x d_input
    d_state=16,
    dt_rank=math.ceil(256 / 16),  # = 16
    n_layer=6,
    vocab_size=len(tokenizer),
    kernel_size=4,
    bias=False,
    conv_bias=True,
)

model = MambaLMHeadModel(config).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Parameters: 15,495,936


Training on SimpleStories dataset

In [ ]:
BLOCK_SIZE = 256
BATCH_SIZE = 16
EPOCHS = 1
LR = 3e-4

class TokenDataset(Dataset):
    def __init__(self, tokens, block_size):
        self.tokens = tokens
        self.block_size = block_size

    def __len__(self):
        return len(self.tokens) // self.block_size

    def __getitem__(self, i):
        chunk = self.tokens[i * self.block_size : (i + 1) * self.block_size + 1]
        return torch.tensor(chunk[:-1], dtype=torch.long), torch.tensor(chunk[1:], dtype=torch.long)

print("Loading TinyStories dataset...")
raw = load_dataset("roneneldan/TinyStories", split="train[:5%]")  # 5% for quick test

print("Tokenizing...")
text = "\n".join(raw["text"])
tokens = tokenizer.encode(text)
print(f"Total tokens: {len(tokens):,}")

dataset = TokenDataset(tokens, BLOCK_SIZE)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn = torch.nn.CrossEntropyLoss()

print(f"\nTraining for {EPOCHS} epoch(s), {len(loader)} batches...")
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for step, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        logits = model(x).logits
        loss = loss_fn(logits.reshape(-1, config.vocab_size), y.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        if step % 50 == 0:
            print(f"  step {step}/{len(loader)}  loss={loss.item():.4f}")
    print(f"Epoch {epoch+1} avg loss: {total_loss / len(loader):.4f}")

Generate

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=200, temperature=0.8, device="cuda"):
    model.eval()
    input_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long).unsqueeze(0).to(device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(input_ids).logits[:, -1, :]
            logits = logits / temperature
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)
    return tokenizer.decode(input_ids[0].tolist())

prompt = "Once upon a time"
print(f"Prompt: {prompt!r}\n")
print(generate(model, tokenizer, prompt, max_new_tokens=200, temperature=0.8, device=device))